In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l6.7 MoE introduction

> 🎯 **Goal:** Meet the Mixture of Experts idea: many MLPs plus a router, so the model
gains capacity without paying proportionally more compute per token.

**Why this matters.** This is how several of the largest deployed models (Mixtral,
and reportedly GPT-4-class systems) get enormous parameter counts while keeping the
per-token cost manageable. It is the lever that decouples *capacity* from *compute*,
the last big architectural idea in this unit.

**The intuition.** A normal transformer block has one MLP that processes every token.
A Mixture of Experts (MoE) replaces that single MLP with many parallel experts and
adds a small *router* that, for each token, picks just a few experts to actually run.
Think of a panel of specialists: every patient (token) sees only the one or two
specialists relevant to them, not the whole hospital. You get the knowledge of the
entire panel available, but each visit only pays for a couple of doctors.

**The mechanics.** Say there are `n_experts` expert MLPs. The router is a tiny linear
layer that scores the experts for each token; top-1 routing sends the token to its
single highest-scoring expert (real systems use top-k, typically top-2). Only the
chosen experts run, so the *compute* per token stays close to one MLP even though the
model's total *parameters* grow with the number of experts. That is the trade: huge
capacity (many experts to specialize) at roughly constant per-token cost (sparse
activation).

In [ ]:
from torch import nn

torch.manual_seed(0)
n_experts, dim = 4, 8
experts = nn.ModuleList([nn.Linear(dim, dim) for _ in range(n_experts)])
router = nn.Linear(dim, n_experts)
x = torch.randn(5, dim)
choice = router(x).argmax(-1)   # top-1 routing
out = torch.stack([experts[choice[i]](x[i]) for i in range(len(x))])
print("each token routed to one expert:", choice.tolist())
print("output shape:", tuple(out.shape))

**What just happened.** The router scored 4 experts for each of the 5 tokens, and
`argmax` picked the single best expert per token (top-1). Each token then passed
through only its chosen expert, and the output kept the same shape as the input. That
is sparsity in action: 4 experts' worth of capacity sits in the model, but each token
paid for exactly one.

⚠️ **Common pitfalls**
- The toy loop runs experts one token at a time, which is clear but slow. Real MoE
  kernels gather tokens per expert and run them in batches; never ship the Python loop.
- Load imbalance. If the router sends most tokens to one favorite expert, the others
  starve and the capacity is wasted. Production MoEs add a load-balancing loss to keep
  every expert busy.
- Counting all experts as active compute. Only the routed experts run, so total
  parameters and active compute are different numbers; mixing them up overstates the
  cost.

🏋️ **Try it yourself.** Switch from top-1 to top-2 routing: take the two highest
router scores per token, run both experts, and combine their outputs (a softmax over
the two scores makes a natural weighted sum). Notice how compute per token roughly
doubles while capacity is unchanged, the dial MoE designers actually turn.

In [ ]:
assert tuple(out.shape) == (5, dim)
assert int(choice.max()) < n_experts